In [2]:
import torch

# Number of GPUs available
num_gpus = torch.cuda.device_count()
print(f"Number of GPUs available: {num_gpus}")

Number of GPUs available: 2


In [3]:
import numpy as np
Test_data = np.load('test_data.npy', allow_pickle=True)

for i in Test_data[:100]:
    print(i[3], i[1], i[2])


Chloe Petts is a stand-up comedian from Sittingbourne in Kent. She began as part of the Soho Theatre Young Company in 2015 and has since risen through the comedy ranks at a pleasant speed where she garnered praise in the final of the Leicester Square New Comedian of the Year Award 2017 and came runner-up in the 2017 Funny Women Awards. She has been selected for the prestigious Pleasance Comedy Reserve 2018. comedian F
Carlotta Ferrari is an Italian composer with an international reputation. She is especially known for her choral compositions and organ works. Ferrari combines Renaissance and Medieval traditions within contemporary musical frameworks. composer F
Barbara Heller is a contemporary composer based in Darmstadt, Germany. She is a founding member of the international working group for Women and Music. Written in 1989, Lalai, Lullaby for Awakening? , was dedicated to “all women who are held in Islamic prisons for political reasons, to all those who have already been executed and

In [3]:


import numpy as np
torch.cuda.set_device(1)
ICE_data = np.load('ice_data.npy', allow_pickle=True)
Train_data = np.load('train_data.npy', allow_pickle=True)
Val_data = np.load('val_data.npy', allow_pickle=True)
Test_data = np.load('test_data.npy', allow_pickle=True)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
idx_train = pd.read_csv('train_idx_for_prompt_sel_from_ICE.csv')
idx_val = pd.read_csv('val_idx_for_prompt_sel_from_ICE.csv')
idx_test = pd.read_csv('test_idx_for_prompt_sel_from_ICE.csv')


base_template = """The list of possible professions are:
[ psychologist, poet, photographer, nurse, software_engineer, comedian, pastor, architect, chiropractor, dentist, model, interior_designer, teacher, accountant, rapper, yoga_teacher, paralegal, surgeon, painter, composer, dj, personal_trainer, physician, journalist, dietitian, filmmaker, attorney, professor,]

Examples:
"""

def put_example(bio, prof, gend, flip=0):
    if flip == 1:
        if gend == 'M': gender = 'Female'
        else: gender = 'Male'
    else:
        if gend == 'M': gender = 'Male'
        else: gender = 'Female'
    template = """
Bio: {b1}
Q: Predict the profession and the gender of the above bio
A: {p1}, {g1}
"""
    result_string = template.format(b1=bio, p1=prof, g1=gender)
    
    return result_string

def put_example_test(bio, prof, gend, flip=0):
    if flip == 1:
        if gend == 'M': gender = 'Female'
        else: gender = 'Male'
    else:
        if gend == 'M': gender = 'Male'
        else: gender = 'Female'
    template = """
Now, predict for this given example
Bio: {b1}
Q: Predict the profession and the gender of the above bio
 ### Model Output: {p1}, {g1}
"""
    result_string = template.format(b1=bio, p1=prof, g1=gender)
    
    return result_string

def put_example_test_for_test_data(bio):
    template = """
Now, predict for this given example
Bio: {b1}
Q: Predict the profession and the gender of the above bio
 ### Model Output: """
    result_string = template.format(b1=bio)
    
    return result_string


import ast
from datasets import Dataset, DatasetDict
def get_prompt(idx, data, Test=0):
    result = []
    temp = []
    target = []
    for index, row in idx.iterrows():
        temp_template = base_template
        indices_list = ast.literal_eval(row['indices'])
        for i in indices_list:
           example = put_example(ICE_data[i][3], ICE_data[i][1], ICE_data[i][2])
           temp_template = temp_template + example
        
        if Test == 0:
            temp_template = temp_template + put_example_test(data[index][3], data[index][1], data[index][2], flip=0)
        else:
            temp_template = temp_template + put_example_test_for_test_data(data[index][3])
        temp.append(temp_template)
        gender = 'Male' if data[index][2] == 'M' else 'Female'
        target.append((f'{data[index][1]}, {gender}'))
        result.append(f"{row['num_samples']}, {row['type']}")

    return {'input_text': temp, 'target_text': target, 'combination': result}
            
            
            
train_data = get_prompt(idx_train, Train_data)
val_data = get_prompt(idx_val, Val_data)
test_data = get_prompt(idx_test, Test_data, Test=1)


train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)
val_df = pd.DataFrame(val_data)

train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)
val_df = val_df.sample(frac=1, random_state=42).reset_index(drop=True)


#Convert DataFrames to Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Create DatasetDict
dataset_dict = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})


In [ ]:
from peft import LoraConfig, PeftConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub.hf_api import HfFolder
import torch
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)




model_id = "meta-llama/Llama-2-7b-chat-hf"

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map={"": "cuda:1"}, trust_remote_code=True,)

model.config.use_cache = True # silence the warnings
# model.config.pretraining_tp = 1
# model.gradient_checkpointing_enable()
# model.enable_input_require_grads()
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
#model.resize_token_embeddings(len(tokenizer))
#tokenizer.add_tokens(new_tokens)
#model.resize_token_embeddings(len(tokenizer))


model = PeftModel.from_pretrained(model, './fine_tuned_llama2_test_custom_loss_r8_a64_tok_32000_new_format_loss_v1_best_prompt')


tokenizer.padding_side = "left"
tokenizer.pad_token_id = tokenizer.bos_token_id
tokenizer.pad_token = tokenizer.bos_token
model.config.pad_token_id = tokenizer.bos_token_id
model = model.bfloat16()
#model.resize_token_embeddings(len(tokenizer))


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
# torch.backends.cuda.enable_mem_efficient_sdp(False)
# torch.backends.cuda.enable_flash_sdp(False)

In [5]:
from peft import PeftModel

# Load the fine-tuned model
#model = PeftModel.from_pretrained(model, 'fine_tuned_llama2_test')

# Access the PEFT config to check rank and alpha values
peft_config = model.peft_config

for layer_name, config in peft_config.items():
    print(f"Layer: {layer_name}")
    print(f"LoRA Rank: {config.r}")
    print(f"LoRA Alpha: {config.lora_alpha}")

Layer: default
LoRA Rank: 8
LoRA Alpha: 64


In [6]:
import torch
from tqdm import tqdm  # Import tqdm for progress bar

# Ensure you're in evaluation mode
model.eval()

# Define the batch size
batch_size = 4  # Adjust based on your GPU memory

def infer_batch(model, tokenizer, input_texts, batch_size, max_input_length):
    predictions = []
    num_batches = (len(input_texts) + batch_size - 1) // batch_size  # Calculate number of batches

    with torch.no_grad():
        for i in tqdm(range(num_batches), desc="Processing Batches", unit="batch"):
            start_idx = i * batch_size
            end_idx = min(start_idx + batch_size, len(input_texts))
            batch = input_texts[start_idx:end_idx]
            
            # Tokenize the batch
            inputs = tokenizer(batch, return_tensors="pt", padding="max_length", truncation=True, max_length=max_input_length)
            #print(inputs['input_ids'])
            input_ids = inputs["input_ids"].to(model.device)
            attention_mask = inputs["attention_mask"].to(model.device)
            
            # Forward pass
            outputs = model.generate(input_ids, attention_mask=attention_mask, max_new_tokens=12, num_return_sequences=1)
            #prob = model(inputs[0])
            #print(outputs.tolist())
            # Decode the predictions
            batch_predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            for pred in batch_predictions:
                if 'Model Output:' in pred:
                    #print(pred)
                    _, result = pred.split('Model Output:', 1)  # Split and get part after the delimiter
                    predictions.append(result.strip())
                else:
                    predictions.append(pred.strip())
    
    return predictions

# Test dataset
test_data = test_dataset['input_text']
# Get predictions
predictions = infer_batch(model, tokenizer, test_data, batch_size, max_input_length=2048)
#np.save('custom_loss_pred.npy', np.array(predictions))
# Optionally, compare with the true labels if needed
test_labels = test_dataset['target_text']


Processing Batches:   0%|          | 0/700 [00:00<?, ?batch/s]

Processing Batches: 100%|██████████| 700/700 [36:28<00:00,  3.13s/batch]


In [7]:
predictions

['Profession: interior_designer\nGender',
 'Profession: paralegal\nGender:',
 'Profession: dentist\nGender: Male',
 'Profession: paralegal\nGender:',
 'Profession: accountant\nGender: Male',
 'profession: personal_trainer\ngender: Fem',
 'Predicted profession: interior_designer\nPredicted',
 'profession: attorney, gender: Male',
 'profession: chiropractor\ngender: Male',
 'profession: DJ, gender: Male',
 'profession: accountant\ngender: Female',
 '100%\nProfession: Adult Model',
 'profession: software_engineer\ngender: Male',
 'profession: orthopedic_surgeon\ngender',
 'Profession: spiritual teacher\nGender: Male',
 'Profession: accountant\nGender: Male',
 'A: architect, Male',
 'Profession: dentist\nGender: Male',
 'Profession: psychologist\nGender: Fem',
 'Profession: psychologist, Female',
 'profession: painter, Male',
 'profession: professor\ngender: Male',
 'Profession: comedian\nGender:',
 'profession: painter\ngender: Male',
 'profession: accountant\ngender: Male',
 'profession:

In [8]:
cleaned_res = [item.split('\n')[0] for item in predictions]
split_data_generated = [item.split(', ') for item in cleaned_res]
split_data_true = [item.split(', ') for item in test_labels]


In [ ]:
gc = 0


gen_gend = []
true_gend = []
temp = []

prof_to_match = [
    'psychologist', 'poet', 'photographer', 'nurse', 'software_engineer',
    'comedian', 'pastor', 'architect', 'chiropractor', 'dentist', 'model',
    'interior_designer', 'teacher', 'accountant', 'rapper', 'yoga_teacher',
    'paralegal', 'surgeon', 'painter', 'composer', 'dj', 'personal_trainer',
    'physician', 'journalist', 'dietitian', 'filmmaker', 'attorney', 'professor'
]
gend_to_match = ['Male', 'Female']
for i, txt in enumerate(predictions):
    flag = 0
    for gend in gend_to_match:
        if gend.lower() in txt.lower() and flag == 0:
            #print(gend, split_data_true[i][1])
            if(gend.lower()==split_data_true[i][1].lower() or (gend.lower()=='fem' and split_data_true[i][1] == 'Female') or (gend.lower()=='feale' and split_data_true[i][1] == 'Female')):
                gc+=1
                gen_gend.append(split_data_true[i][1])
                temp.append(gend)
                true_gend.append(split_data_true[i][1])
                flag = 1
                break
    if flag == 0: 
        
        gen_gend.append('Male' if split_data_true[i][1] == 'Female' else 'Female')
        true_gend.append(split_data_true[i][1])
print(gc/len(predictions))
    

In [10]:
from sentence_transformers import SentenceTransformer, util
bert_model = SentenceTransformer('all-MiniLM-L6-v2')

professions_to_match = [
    'psychologist', 'poet', 'photographer', 'nurse', 'software_engineer',
    'comedian', 'pastor', 'architect', 'chiropractor', 'dentist', 'model',
    'interior_designer', 'teacher', 'accountant', 'rapper', 'yoga_teacher',
    'paralegal', 'surgeon', 'painter', 'composer', 'dj', 'personal_trainer',
    'physician', 'journalist', 'dietitian', 'filmmaker', 'attorney', 'professor'
]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [11]:
profession_correct = 0
#gender_correct = 0
c = 0
true_prof = []
gen_prof = []

i = 0

for gen, true in zip(split_data_generated, split_data_true):
    print(i)
    i+=1
    flag = 0
    true_prof.append(true[0])
    #true_gend.append(true[1])
    temp = 'NA'
    if len(gen) < 2: 
        if (isinstance(gen, list)): gen = gen[0]
        gen_profession = gen
        true_profession, _ = true
        #print(gen, true)
        c+=1
        # gen_prof.append('NA')
        # if true[1] == 'Male':
        #     gen_gend.append('Female')
        # else:
        #     gen_gend.append('Male')
    elif(len(gen) > 2):
        #print(gen)
        gen = gen[:2]
        
        gen_profession, _ = gen
        true_profession, _ = true
    else: 
        gen_profession, _ = gen
        true_profession, _ = true
    
    #print(true_gender, gen_gender)
    # Profession check
    if true_profession.lower() in gen_profession.lower():
        gen_prof.append(true_profession)
        profession_correct += 1
        flag = 1
    elif(len(gen_profession)!= 0):
        print(true_profession,'|', gen_profession)
        
        if gen_profession not in professions_to_match:
            similarity_array=[]
            for profession in professions_to_match:
                word1=gen_profession
                word2=profession
                embeddings1 = bert_model.encode(word1, convert_to_tensor=True)
                embeddings2 = bert_model.encode(word2, convert_to_tensor=True)
                similarity_array.append((util.cos_sim(embeddings1, embeddings2).item(),profession))
            sorted_similarity_array = sorted(similarity_array, key=lambda x: x[0], reverse=True)
            temp = sorted_similarity_array[0][1]
            #print(sorted_similarity_array[0][1])
            if sorted_similarity_array[0][1] == true_profession.lower():
                #print(sorted_similarity_array[0][1], gen_profession)
                gen_prof.append(true_profession)
                profession_correct += 1
                flag = 1
        else:
            temp = gen_profession

    if flag == 0:
        #print(true_profession, gen_profession)
        gen_prof.append(temp)
        #print(true_profession,'|', gen_profession, '|', temp)

    # Gender check
    # if ': Male' in gen_gender or ': Fem' in gen_gender:
    #     gen_gender = gen_gender.split(': ')[1]
    #     #print(gen_gender)
    # if gen_gender == true_gender or (gen_gender == 'Fem' and true_gender == 'Female'):
    #     gender_correct += 1
    #     #print(gen_gender, true_gender)
    #     gen_gend.append(true_gender)
        
    # else:
    #     #print(gen_gender, true_gender)
    #     if true[1] == 'Male':
    #         gen_gend.append('Female')
    #     else:
    #         gen_gend.append('Male')


# Total number of samples
total_samples = len(split_data_true)

# Calculate accuracy
profession_accuracy = profession_correct / total_samples
#gender_accuracy = gender_correct / total_samples

print(f"Profession Accuracy: {profession_accuracy:.5f}")
#print(f"Gender Accuracy: {gender_accuracy:.5f}")

0
1
2
3
4
5
6
7
8
9
10
11
model | 100%
12
13
14
15
16
17
18
19
professor | Profession: psychologist
20
21
22
23
24
25
26
27
28
29
30
31
32
33
dietitian | Profession: nutritionist
34
35
36
37
38
39
model | 100
40
41
42
43
44
45
dj | profession: model
46
47
48
architect | profession: software_engineer
49
50
51
52
software_engineer | Profession: author
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
psychologist | Profession: psychotherapist
68
69
70
71
72
73
74
75
attorney | Profession: lawyer
76
77
78
journalist | profession: writer
79
80
dietitian | Profession: nutritionist
81
82
83
84
85
86
87
88
89
90
91
92
93
94
professor | profession: physician
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
model | Female
118
119
120
121
122
123
124
125
126
comedian | Profession: poet
127
128
129
130
131
132
133
134
135
136
architect | profession: software_engineer
137
138
composer | profession: producer
139
140
surgeon | profession: physician
141
142
143
144
14

In [ ]:
import pandas as pd

# Assuming true_prof, gen_prof, true_gend, gen_gend are lists or arrays
data = {
    'True_Profession': true_prof,
    'Generated_Profession': gen_prof,
    'True_Gender': true_gend,
    'Generated_Gender': gen_gend
}

# Create a DataFrame
df = pd.DataFrame(data)

# Save DataFrame to a CSV file
df.to_csv('./predictions/fine_tuned_llama2.csv', index=False)


In [2]:
import pandas as pd

df = pd.read_csv('./predictions/profession_gender_predictions_format_loss_v1.csv')

# Convert the columns back into lists
true_prof = df['True_Profession'].tolist()
gen_prof = df['Generated_Profession'].tolist()
true_gend = df['True_Gender'].tolist()
gen_gend = df['Generated_Gender'].tolist()

In [3]:
# Calculate the accuracy for professions
profession_accuracy = sum([t == g for t, g in zip(true_prof, gen_prof)]) / len(true_prof)

# Calculate the accuracy for genders
gender_accuracy = sum([t == g for t, g in zip(true_gend, gen_gend)]) / len(true_gend)

# Display the accuracies
print(f"Profession Accuracy: {profession_accuracy:.2%}")
print(f"Gender Accuracy: {gender_accuracy:.2%}")


Profession Accuracy: 99.96%
Gender Accuracy: 61.50%
